# TME 310 coding assignment 04

## Instructions

1. Solve each problem in the code cell below it. Add cells if you need them.
2. Some problems end with a task that begins "In a markdown cell". Answer it in the cell headed **Your explanation** under the code, replacing the placeholder text. A comment inside a code cell does not count.
3. If a problem takes more than 30 minutes, stop and get help: the [course AI tutor](https://uwt-tme-310.github.io/TME_310_A/syllabus/polya_ai.html), office hours, or your group.
4. When you finish, restart the kernel, run all cells, export the notebook as HTML or PDF, and upload it on Canvas.

## Learning objectives

- Create, validate, and use models of physical systems
- Use code-based numerical tools

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

## Problem 1

The "divide and average" method, an old-time method for approximating the square root of any positive number $a$, can be formulated as:

$x_{new} = \displaystyle \frac{x + a/x}{2}$

The algorithm, depicted in the flowchart below, repeats this step until the **approximate relative error** is less than some pre-defined tolerance.

<img src="sqrt_approx.png" width="400">

---

Write a function `sqrt_approx(a, tol=1e-6)` that approximates the square root of any positive number using the divide and average method, stopping when the approximate relative error is below `tol`.
Test it on 2, 4, 9, 100 and 250, and print your results next to `np.sqrt()` for comparison.



In [ ]:
# Your code here

## Problem 2

In the last assignment a slope's resistance to sliding was modeled with friction alone and a 60° slope could not stand. 
Many soils also resist failure through **cohesion**: shear strength that does not depend on the normal stress. 
The Mohr-Coulomb strength model combines the two:

$$\tau_f = c + \sigma_n \tan\phi$$

where $\tau_f$ is the shear strength (kPa), $c$ the cohesion (kPa), $\sigma_n$ the normal stress on the plane (kPa) and $\phi$ the friction angle. 

In the last assignment, we also modeled the slope as a single sliding mass sitting on an inclined plane.
In this problem, we will break the sliding mass into segments and calculate the driving and resisting forces on each segment.
(NOTE: breaking things into segments like this is called **discretization**.)

The normal stress is no longer the same everywhere on the plane: it comes from the weight of the soil column above each point,

$$\sigma_n(x) = \gamma\, h(x) \cos^2\theta$$

with $x$ the horizontal distance from the toe and $h(x)$ the height of soil above the plane at that $x$. Under the slope face $h$ grows with $x$; past the crest the ground is flat and $h$ shrinks to zero where the plane reaches the surface.

Because the strength varies along the plane, the wedge is cut into $n$ vertical **slices** of equal width $\Delta x$, and each slice is treated like a small block on the plane. For slice $i$ with midpoint $x_i$ and column height $h_i = h(x_i)$:

* Slice weight: $W_i = \gamma\, h_i\, \Delta x$
* Driving force: $D_i = W_i \sin\theta$
* Length of plane under the slice: $\Delta l = \Delta x / \cos\theta$
* Resisting force: $R_i = \tau_f(\sigma_{n,i})\, \Delta l$, with $\sigma_{n,i} = \gamma\, h_i \cos^2\theta$

The whole wedge's driving and resisting forces are the sums over the slices, and $FS = \sum R_i \big/ \sum D_i$. This is the same model as the [slope stability project](https://uwt-tme-310.github.io/TME_310_A/extra_labor/proj_slope-stability.html), which uses it for soil whose strength changes from layer to layer.

<img src="discretization.png" width="60%">

Same slope as before: $H = 10$ m, $\beta = 60°$, $\gamma = 19$ kN/m³. The soil has $c = 15$ kPa and $\phi = 30°$. Use $\theta = 45°$ and $n = 10$ slices throughout.

---

1. Complete `slice_heights(theta_deg, n=10, H=10, beta_deg=60)` below so it returns three arrays of length `n`: the slice midpoints `x`, the column heights `h`, and (as a number) the slice width `dx`. Build the slice edges with `np.linspace(0, x_top, n + 1)` and take their midpoints; use a mask on `x` to compute $h$ under the face ($h = x\tan\beta - x\tan\theta$) and past the crest ($h = H - x\tan\theta$).
2. Write `slice_forces(theta_deg, n=10, c=15, phi_deg=30, H=10, beta_deg=60, gamma=19)` that calls `slice_heights` and returns the arrays `D` and `R` of driving and resisting force per slice (kN per metre of width).
3. Plot $h$ against $x$ in one figure and $D_i$ and $R_i$ against $x$ together in a second figure, with labels and a legend.
4. Print the total driving force, the total resisting force and $FS$. As a check, print the sum of the slice weights next to the wedge weight from the last assignment, $W = \gamma A$ with $A = \frac{H^2}{2}(\cot\theta - \cot\beta)$; they should agree to within about a percent.


In [ ]:
import numpy as np

def slice_heights(theta_deg, n=10, H=10, beta_deg=60):
    theta = np.radians(theta_deg)
    beta = np.radians(beta_deg)
    x_top = H / np.tan(theta)        # where the plane reaches the crest level
    x_crest = H / np.tan(beta)       # horizontal position of the crest
    # Your code here
    return x, h, dx

In [ ]:
# Your code here

## Problem 3

The previous problem cut the wedge into 10 slices without asking whether 10 is enough. For uniform Mohr-Coulomb soil there is a closed form to compare against,

$$R = cL + W\cos\theta\tan\phi, \qquad L = \frac{H}{\sin\theta}, \qquad D = W\sin\theta$$

with $W = \gamma A$ and $A = \frac{H^2}{2}(\cot\theta - \cot\beta)$ as before. This problem uses `slice_forces` from the previous one (copy it into the cell below if you are working in a fresh notebook). Same slope and soil, $\theta = 45°$.

---

1. Write `fs_sliced(theta_deg, n, c=15, phi_deg=30, H=10, beta_deg=60, gamma=19)` that calls `slice_forces` and returns $FS = \sum R_i / \sum D_i$.
2. With a loop over $n$ = 10, 100 and 1000, print a table of $n$, $FS$, and the approximate relative error between successive values of $n$.
3. Plot $FS$ against $n$ with a logarithmic horizontal axis (`ax.set_xscale("log")`), and add a horizontal line at the closed-form $FS$.
4. Print the true relative error of $FS$ for each $n$ against the closed form.
5. In a markdown cell: for this soil the closed form gives the answer in one line, so why is the sliced version worth having? Read the [slope stability project](https://uwt-tme-310.github.io/TME_310_A/extra_labor/proj_slope-stability.html) page and say what changes about the soil there that the closed form cannot handle and the slices can.

In [ ]:
# Your code here

### Your explanation

Replace this text with your answer to the last task of the problem (the one that begins "In a markdown cell"). Write it here, in this markdown cell, not as a comment in a code cell.

## Extra problem (optional)

*This problem is an [extra labor](https://uwt-tme-310.github.io/TME_310_A/extra_labor/). It is not required for the assignment to be complete. Completing it adds 0.1 to your grade (extra problems count up to 0.3 in total), and you can submit it any time before the end of the quarter. Expect it to take about two hours.*

Two classic series converge to $\pi$:

* Leibniz: $\displaystyle \pi = 4\left(1 - \frac{1}{3} + \frac{1}{5} - \frac{1}{7} + \cdots\right)$
* Nilakantha: $\displaystyle \pi = 3 + \frac{4}{2\cdot3\cdot4} - \frac{4}{4\cdot5\cdot6} + \frac{4}{6\cdot7\cdot8} - \cdots$

---

1. Write a function for each series that adds terms in a `while` loop until the **approximate relative error** falls below a tolerance passed as an argument, and returns the estimate, the number of terms used, and arrays of the estimate after each term.
2. Run both with a tolerance of $10^{-6}$. How many terms does each need?
3. Plot the **true relative error** of each series against the number of terms on log-log axes (`ax.set_xscale("log")`, `ax.set_yscale("log")`), using `np.pi` as the true value.
4. For each series, print the approximate relative error at the point your loop stopped next to the true relative error there.
5. In a markdown cell: which series converges faster, and why? Is the approximate relative error a good stand-in for the true error here, when it was not for the dartboard estimate of π from class?

In [ ]:
# Your code here

### Your explanation

Replace this text with your answer to the last task of the problem (the one that begins "In a markdown cell"). Write it here, in this markdown cell, not as a comment in a code cell.